# Module 1: The Sandbox (MuJoCo & Headless Simulation)

Imagine you have a magic, invisible box. Inside this box, gravity exists, objects have weight, and things crash into each other. But none of it is real—it is entirely made of math. 

This is **MuJoCo** (Multi-Joint dynamics with Contact). 

When we train a robot like the Microduck, we don't start with physical metal and plastic. If a physical robot falls 10,000 times while learning to walk, its motors will burn out and its 3D-printed joints will snap. In our MuJoCo "Sandbox," the robot can fall a million times in an hour, and it costs us nothing.

### The Two Halves of the Universe

To make this invisible sandbox work, MuJoCo splits the universe into two pieces:

1. **The Model (`MjModel`):** The blueprint. This is the immutable physics rules of your world. It defines what gravity is, how much the duck weighs, and the shape of its legs. Once the simulation starts, the Model *never changes*.
2. **The Data (`MjData`):** The live state. This holds the numbers that constantly change, like the exact height of the duck's head, or how fast it is currently falling. 

```text
+-------------------+       +-------------------+
|   MjModel (XML)   |       |   MjData (State)  |
|-------------------|       |-------------------|
| - Gravity: 9.81   |       | - Duck Z: 1.00m   |
| - Mass: 0.8kg     | ----> | - Velocity: 0 m/s |
| - Shapes: Capsule |       | - Joints: 0 deg   |
+-------------------+       +-------------------+

In [1]:
import mujoco

# 1. The Blueprint (XML)
# We define a floor (plane) and a 0.8kg capsule (our mock duck body) floating 1 meter up (pos="0 0 1").
xml_string = """
<mujoco>
  <worldbody>
    <geom type="plane" size="1 1 0.1" rgba=".9 0 0 1"/>
    <body name="duck_body" pos="0 0 1">
      <joint type="free"/>
      <geom type="capsule" size="0.1 0.2" rgba="1 .8 .2 1" mass="0.8"/>
    </body>
  </worldbody>
</mujoco>
"""

# 2. Create the Universe
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)

print(f"🌍 Universe created. Gravity is set to: {model.opt.gravity}")
print(f"🦆 Starting altitude (Z-axis): {data.qpos[2]} meters")

🌍 Universe created. Gravity is set to: [ 0.    0.   -9.81]
🦆 Starting altitude (Z-axis): 1.0 meters


### The Tick of Time (`mj_step`)

Nothing happens in MuJoCo until we tell time to move forward. We do this by calling `mujoco.mj_step(model, data)`. 

Every time we call this function, the physics engine looks at the **Model** (gravity is pulling down) and updates the **Data** (the duck is now lower and moving faster). One "step" is usually 2 milliseconds of simulated time.

In [2]:
print("Letting go of the duck...")
print("-" * 40)

# Step time forward 10 times
for i in range(1, 11):
    # This is the heartbeat of Physical AI!
    mujoco.mj_step(model, data)
    
    # qpos[2] is position (height). qvel[2] is velocity (speed of falling).
    height = data.qpos[2]
    speed = data.qvel[2] 
    
    print(f"Tick {i:02d} | Height: {height:.4f}m | Falling Speed: {speed:.4f} m/s")

Letting go of the duck...
----------------------------------------
Tick 01 | Height: 1.0000m | Falling Speed: -0.0196 m/s
Tick 02 | Height: 0.9999m | Falling Speed: -0.0392 m/s
Tick 03 | Height: 0.9998m | Falling Speed: -0.0589 m/s
Tick 04 | Height: 0.9996m | Falling Speed: -0.0785 m/s
Tick 05 | Height: 0.9994m | Falling Speed: -0.0981 m/s
Tick 06 | Height: 0.9992m | Falling Speed: -0.1177 m/s
Tick 07 | Height: 0.9989m | Falling Speed: -0.1373 m/s
Tick 08 | Height: 0.9986m | Falling Speed: -0.1570 m/s
Tick 09 | Height: 0.9982m | Falling Speed: -0.1766 m/s
Tick 10 | Height: 0.9978m | Falling Speed: -0.1962 m/s


# Module 2: The Gym (Reinforcement Learning & PPO)

If you want a dog to sit, you don't grab its legs, calculate joint angles, and manually bend its knees. 
Instead, you hold up a treat, wait for the dog to figure out the muscle movements, and then give the treat when it succeeds.

This is **Reinforcement Learning (RL)**. We don't write code telling the robot *how* to balance. We build a virtual room (a "Gym") with a reset button, and we give the AI "treats" (points) when it stays upright.

### The Gym Rules (State, Action, Reward)

To train our network using the PPO (Proximal Policy Optimization) algorithm, our Gym must follow three strict rules:

| Rule                        | The Dog Analogy                    | The Python Equivalent                                 |
| :-------------------------- | :--------------------------------- | :---------------------------------------------------- |
| **1. Observation Space**    | What the dog sees and hears.       | `observation_space` (60 float values for sensors).    |
| **2. Action Space**         | What muscles the dog can move.     | `action_space` (15 float values between -1.0 and 1.0).|
| **3. Reward**               | The dog biscuit.                   | `reward` (+1 point for staying up, 0 for falling).    |

When we wrap our MuJoCo physics engine in a Gymnasium class, we are just translating gravity and collisions into these three rules so the AI can understand them.

In [3]:
import numpy as np
import gymnasium as gym

class MiniMicroduckGym(gym.Env):
    def __init__(self):
        super().__init__()
        # ACTION: 15 motors. We strictly constrain the AI to hardware limits.
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(15,), dtype=np.float32)
        
        # OBSERVATION: 60 sensor readings.
        self.observation_space = gym.spaces.Box(low=-50.0, high=50.0, shape=(60,), dtype=np.float32)
        
        self.duck_height = 1.0  # Starting height
        
    def reset(self, seed=None, options=None):
        """The 'Reset Button'. Called when the duck falls over."""
        super().reset(seed=seed)
        self.duck_height = 1.0
        return np.zeros(60, dtype=np.float32), {} # Return empty sensors and info

    def step(self, action):
        """The AI makes a move, we calculate the physics results."""
        # Simulated physics: The duck slowly falls unless the motors balance it
        self.duck_height -= 0.05 
        
        # REWARD LOGIC: Did it fall?
        terminated = bool(self.duck_height <= 0.5) # Duck faceplants at 0.5m!
        reward = 0.0 if terminated else 1.0        # Treat for staying up!
        
        # Give back the next mock sensor reading
        mock_sensors = np.random.uniform(-0.1, 0.1, 60).astype(np.float32)
        
        return mock_sensors, reward, terminated, False, {}

# Test our Gym!
env = MiniMicroduckGym()
print("🏋️ Gym built. Spaces defined:")
print(f"   Actions:    {env.action_space.shape[0]} motors")
print(f"   Sensors:    {env.observation_space.shape[0]} inputs")

🏋️ Gym built. Spaces defined:
   Actions:    15 motors
   Sensors:    60 inputs


### The Training Loop

When we pass this `env` to the Stable Baselines3 PPO algorithm, PPO acts like an ultra-fast dog trainer. It hits `reset()`, fires random actions into `step()`, and watches the reward. 

It does this *millions* of times. Over time, the neural network inside PPO mathematically shifts its weights to maximize the score, discovering the concept of "balance" entirely by accident.

In [4]:
print("🐕 Simulating the RL Trainer testing the environment...")
env.reset()

for tick in range(1, 15):
    # The untrained AI guesses random motor movements
    random_action = env.action_space.sample() 
    
    # We pass the action into the gym
    state, reward, terminated, truncated, info = env.step(random_action)
    
    status = "💀 FELL OVER (Terminated!)" if terminated else "🦆 Balancing... (+1 Reward)"
    print(f"Tick {tick:02d} | Height: {env.duck_height:.2f}m | {status}")
    
    if terminated:
        print("-" * 45)
        print("Hitting the Reset Button and starting over!")
        break

🐕 Simulating the RL Trainer testing the environment...
Tick 01 | Height: 0.95m | 🦆 Balancing... (+1 Reward)
Tick 02 | Height: 0.90m | 🦆 Balancing... (+1 Reward)
Tick 03 | Height: 0.85m | 🦆 Balancing... (+1 Reward)
Tick 04 | Height: 0.80m | 🦆 Balancing... (+1 Reward)
Tick 05 | Height: 0.75m | 🦆 Balancing... (+1 Reward)
Tick 06 | Height: 0.70m | 🦆 Balancing... (+1 Reward)
Tick 07 | Height: 0.65m | 🦆 Balancing... (+1 Reward)
Tick 08 | Height: 0.60m | 🦆 Balancing... (+1 Reward)
Tick 09 | Height: 0.55m | 🦆 Balancing... (+1 Reward)
Tick 10 | Height: 0.50m | 💀 FELL OVER (Terminated!)
---------------------------------------------
Hitting the Reset Button and starting over!


### Deep Dive: What exactly is "Stable Baselines3 PPO"?

In the last cell, our robot guessed random motor movements and immediately fell over. To actually learn, we need an algorithm that remembers what worked and tries to do it better next time. 

That is where **Stable Baselines3** and **PPO** come in.

* **Stable Baselines3 (SB3):** Think of this as the "scaffolding" library. Just like you wouldn't write your own HTTP protocol to build a web app (you'd use FastAPI), you don't write Reinforcement Learning math from scratch. SB3 is a library of pre-packaged, production-ready RL algorithms.
* **PPO (Proximal Policy Optimization):** This is the specific brain architecture we are using. It is the undisputed industry standard for robotics (and also what OpenAI used to train ChatGPT). 

Here is how PPO breaks down, using the analogy of a **Gymnast (The Actor)** and a **Coach (The Critic)**.

#### 1. The Actor-Critic Brain
PPO doesn't just build one neural network; it builds two working together:
* **The Actor (Policy):** Looks at the sensors and fires the motors. It is the reflexes.
* **The Critic (Value):** Looks at the sensors and predicts the score. "You are leaning too far forward, I predict we are going to score zero because we are about to fall."

#### 2. The "Proximal" Safety Net
Imagine the Gymnast accidentally does a perfect backflip. If the Gymnast tries to radically change their technique to do it *twice as good* on the next try, they will probably land on their neck and forget how to do a backflip entirely. 

**Proximal** means "close by." The PPO algorithm has a mathematical safety net that prevents the neural network weights from changing too drastically after a good (or bad) run. It forces the AI to learn in small, safe, incremental steps. 

#### Why this matters for Physical AI (Foreshadowing Module 3)
During training in the MuJoCo simulator, the Actor and the Critic talk to each other millions of times. 
But once the robot is fully trained and we put the chip inside the physical Microduck, **we don't need the Coach anymore.** The robot just needs its reflexes. In the next module, we will literally perform brain surgery to cut out the Critic network before we put it in the robot!

# Module 3: The Brain Surgery (ONNX Export & Clamping)

When training finished in Module 2, Stable Baselines3 handed us a heavy zip file (`microduck_ppo_policy.zip`). 

Inside that zip file is a complex PyTorch system containing both **The Actor** (the Gymnast's reflexes) and **The Critic** (the Coach grading performance). It also contains a bunch of probability math used to guess random variations during training.

If you try to load that entire training setup onto the Microduck's tiny onboard computer:
1. **It wastes compute:** Calculating what the Coach thinks is useless while actually walking down the hall.
2. **It is dangerous:** Raw PPO math produces unbounded numbers (like `+3.42` or `-12.8`). If an unclamped number hits a real motor controller that only accepts `-1.0` to `1.0`, the motor can strip its gears or burn out.

We fix both problems with two surgical steps:

```text
  [ Heavy SB3 Brain ]
   ├── Critic Network   ---> ✂️ (Trash: Only needed for training)
   └── Actor Network    ---> 🛡️ [ HARDWARE CLAMP: min -1.0, max +1.0 ] ---> [ Clean ONNX Silicon File ]

In [5]:
import torch as th
from stable_baselines3 import PPO

# 1. The Surgical Wrapper
class OnnxableActorOnly(th.nn.Module):
    def __init__(self, extractor, action_net):
        super().__init__()
        # We ONLY keep the actor's layers (Feature Extractor + Action Output)
        self.extractor = extractor
        self.action_net = action_net

    def forward(self, observation):
        # Pass the 60 sensory points through the neural network
        action_features, _ = self.extractor(observation)
        raw_motor_guess = self.action_net(action_features)
        
        # 🛡️ HARDWARE SAFETY CLAMP:
        # We bake torch.clamp directly into the computational graph.
        # This guarantees mathematical safety: no value leaving this model
        # can EVER exceed -1.0 or +1.0, no matter what happens to the sensors.
        safe_action = th.clamp(raw_motor_guess, min=-1.0, max=1.0)
        
        return safe_action

# 2. Load the trained SB3 model from disk
sb3_model = PPO.load("microduck_ppo_policy.zip", device="cpu")

# 3. Extract ONLY the Actor components
actor_model = OnnxableActorOnly(
    sb3_model.policy.mlp_extractor,
    sb3_model.policy.action_net
)

print("✂️ Surgery complete! Critic discarded. Hardware safety clamp installed.")

✂️ Surgery complete! Critic discarded. Hardware safety clamp installed.


### Freezing the Reflexes into ONNX

PyTorch models are dynamic Python objects. But Python is slow, heavy, and full of dependencies.

**ONNX (Open Neural Network Exchange)** turns that Python code into a static, frozen file of pure linear algebra. Once exported, the robot doesn't even need PyTorch installed to run its reflexes!

To export to ONNX, PyTorch needs a "dummy input" with the exact shape of our sensors (1 batch × 60 sensory points) so it can trace the mathematical graph from input to output.

In [6]:
# Code cell to export and verify the ONNX file:

# 1. Create dummy sensory input (1 robot, 60 sensor points)
dummy_telemetry = th.randn(1, 60)

# 2. Export the traced graph to an ONNX file
th.onnx.export(
    actor_model,
    dummy_telemetry,
    "microduck_walking_policy.onnx",
    opset_version=17,
    input_names=["observations"],
    output_names=["actions"]
)

print("✅ Saved: 'microduck_walking_policy.onnx'")
print("📦 Model is frozen, stripped of training overhead, and bounded for hardware safety.")

/tmp/ipykernel_9406/3202039947.py:7: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  th.onnx.export(
W0829 15:31:09.569000 9406 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0829 15:31:10.790000 9406 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W0829 15:31:10.793000 9406 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not

[torch.onnx] Obtain model graph for `OnnxableActorOnly([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `OnnxableActorOnly([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/s/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Saved: 'microduck_walking_policy.onnx'
📦 Model is frozen, stripped of training overhead, and bounded for hardware safety.


# Module 4: The Reflex Loop (Temporal Memory & 50Hz Control)

If you look at a photograph of a baseball in mid-air, can you tell which direction it is moving? No. It could be flying toward the batter, or falling straight down. 

To understand momentum and velocity, you need a *sequence* of images. 

Robots face the exact same problem. If the Microduck's brain only looks at a single instant of sensor data (1 frame), it doesn't know if it is falling or standing still. It needs memory.

### The Sliding Window (Muscle Memory)

Earlier in this tutorial, we discovered that giving an AI "infinite memory" (like a chatbot history) causes a token explosion and massive latency. 

Instead, Physical AI uses a **Sliding Window**. 
Imagine a picture frame that can only hold exactly 4 photographs. When a new photograph arrives, you slide it in the right side, and the oldest photograph falls out the left side. 

*   **Fixed Size:** The AI always processes exactly 60 data points (4 frames × 15 sensors). It never grows.
*   **Zero Latency Spikes:** Because the math size is locked, your RTX 5060 GPU can calculate the reflexes in under 2 milliseconds, forever.

In [7]:
# Code to see exactly how Python handles a sliding window using a deque (Double-Ended Queue):

from collections import deque
import numpy as np

# Create a sliding window that holds exactly 4 frames
memory_buffer = deque(maxlen=4)

print("🖼️ Filling the sliding window with mock sensor frames...\n")

# Let's simulate 6 ticks of time
for tick in range(1, 7):
    # Create a mock frame (just a single number for this visual example instead of 15)
    mock_frame = f"Frame_{tick}"
    memory_buffer.append(mock_frame)
    
    print(f"Tick {tick}: Added {mock_frame} -> Current Buffer: {list(memory_buffer)}")
    
print("\nNotice how Frame_1 and Frame_2 were automatically pushed out!")

🖼️ Filling the sliding window with mock sensor frames...

Tick 1: Added Frame_1 -> Current Buffer: ['Frame_1']
Tick 2: Added Frame_2 -> Current Buffer: ['Frame_1', 'Frame_2']
Tick 3: Added Frame_3 -> Current Buffer: ['Frame_1', 'Frame_2', 'Frame_3']
Tick 4: Added Frame_4 -> Current Buffer: ['Frame_1', 'Frame_2', 'Frame_3', 'Frame_4']
Tick 5: Added Frame_5 -> Current Buffer: ['Frame_2', 'Frame_3', 'Frame_4', 'Frame_5']
Tick 6: Added Frame_6 -> Current Buffer: ['Frame_3', 'Frame_4', 'Frame_5', 'Frame_6']

Notice how Frame_1 and Frame_2 were automatically pushed out!


### The 50Hz Heartbeat (explain the final 50Hz execution)

A bipedal robot usually needs to adjust its balance 50 times a second. That means our control loop must run at **50 Hertz (Hz)**. 

$1 \text{ second} / 50 = 0.02 \text{ seconds (20 milliseconds)}$.

Our entire loop (Sense, Think, Act) must complete in under 20 milliseconds. If the ONNX inference takes 2 milliseconds, we literally tell the code to `sleep` for the remaining 18 milliseconds to maintain a perfect, steady 50Hz rhythm.

# Complete, production-ready edge loop 
This brings everything together!

In [8]:
import time
import onnxruntime as ort

print("🔌 Booting the Microduck Edge Controller...")

# 1. Load the frozen silicon-safe brain
session = ort.InferenceSession("microduck_walking_policy.onnx")

# 2. Initialize the 4-frame memory buffer with zeros (so we don't crash on Tick 1)
real_memory_buffer = deque(maxlen=4)
for _ in range(4):
    real_memory_buffer.append(np.zeros(15, dtype=np.float32))

target_hz = 50
tick_time = 1.0 / target_hz

print("⚡ Starting 50Hz Control Loop (Simulating 5 ticks)...\n")

for tick in range(1, 6):
    start_time = time.time()
    
    # [SENSE] Get new telemetry from the physical hardware (mocked here)
    current_sensors = np.random.uniform(-0.1, 0.1, 15).astype(np.float32)
    
    # Slide the new data into memory
    real_memory_buffer.append(current_sensors)
    
    # Flatten the 4 separate frames into one continuous array of 60 numbers
    observation_tensor = np.concatenate(real_memory_buffer).reshape(1, -1)
    
    # [THINK] Pass the 60 numbers into the ONNX runtime
    # This takes < 2ms on edge hardware
    actions = session.run(None, {"observations": observation_tensor})[0]
    
    # [ACT] Send the 15 motor velocities to the physical servos
    velocities = actions[0]
    
    # Formatting the output to show Left Leg, Right Leg, and Neck
    left_leg = [f"{v:>5.2f}" for v in velocities[0:5]]
    print(f"Tick {tick:02d} | Left Leg Targets: [{', '.join(left_leg)}]")
    
    # Sleep to maintain a strict 50Hz (20ms) cycle
    elapsed = time.time() - start_time
    time.sleep(max(0.0, tick_time - elapsed))
    
print("\n✅ Edge loop execution successful. Latency constraints met.")

🔌 Booting the Microduck Edge Controller...
⚡ Starting 50Hz Control Loop (Simulating 5 ticks)...

Tick 01 | Left Leg Targets: [ 0.00, -0.01,  0.04, -0.06,  0.00]
Tick 02 | Left Leg Targets: [-0.04, -0.03,  0.07, -0.05,  0.04]
Tick 03 | Left Leg Targets: [-0.04, -0.01,  0.11, -0.06,  0.00]
Tick 04 | Left Leg Targets: [-0.04,  0.03,  0.06, -0.07, -0.05]
Tick 05 | Left Leg Targets: [-0.06, -0.00,  0.07, -0.01,  0.00]

✅ Edge loop execution successful. Latency constraints met.


---

# Module 5: The Anatomy (URDF & MJCF Blueprints)

To put our virtual brain into a physical body, the MuJoCo physics engine needs to know exactly how the robot is built. It needs to know how heavy the head is, how far the knees can bend, and how powerful the motors are.

In robotics, we define this using an XML file. The industry standard is **URDF** (Unified Robot Description Format), but MuJoCo uses an optimized version called **MJCF**. 

Think of an MJCF file as having three distinct layers:
1. **The Bones (`<joint>`):** Invisible pivot points. They define how parts connect and rotate (e.g., a knee joint that can only bend 90 degrees).
2. **The Flesh (`<geom>`):** The physical shapes and weight (mass, friction, 3D meshes). This is what collides with the floor.
3. **The Muscles (`<actuator>`):** The electrical motors. This section defines the strict power limits of our hardware, ensuring the physics engine behaves exactly like the $399 physical Microduck.

In [9]:
import mujoco

# 1. We construct the core MJCF blueprint for the 15-DOF Microduck
# Notice the <actuator> section: it strictly bounds the motors between -1.0 and 1.0!
microduck_mjcf = """
<mujoco model="microduck">
  <compiler angle="radian"/>
  
  <default>
    <!-- Default physics for all joints and motors -->
    <joint damping="0.1" armature="0.01"/>
    <!-- The hardware clamp: max motor velocity/torque limits -->
    <motor ctrlrange="-1.0 1.0" ctrllimited="true"/> 
  </default>

  <worldbody>
    <!-- The Floor -->
    <geom type="plane" size="2 2 0.1" rgba="0.9 0.9 0.9 1"/>
    
    <!-- The Microduck Pelvis (Root Body) floating 25cm high -->
    <body name="pelvis" pos="0 0 0.25">
      <joint type="free" name="root"/>
      <geom type="box" size="0.04 0.05 0.03" rgba="1 0.8 0.2 1" mass="0.3"/>
      
      <!-- We define the 15 specific joints (Mocked as simple hinge joints for this lesson) -->
      <!-- Left Leg (5) -->
      <body name="l_leg"><joint name="l_hip_yaw" type="hinge" axis="0 0 1"/><geom type="capsule" size="0.01 0.05"/></body>
      <body name="l_knee"><joint name="l_hip_roll" type="hinge" axis="1 0 0"/></body>
      <body name="l_ankle"><joint name="l_hip_pitch" type="hinge" axis="0 1 0"/></body>
      <body name="l_foot"><joint name="l_knee_pitch" type="hinge" axis="0 1 0"/></body>
      <body name="l_toe"><joint name="l_ankle_pitch" type="hinge" axis="0 1 0"/></body>
      
      <!-- Right Leg (5) -->
      <body name="r_leg"><joint name="r_hip_yaw" type="hinge" axis="0 0 1"/><geom type="capsule" size="0.01 0.05"/></body>
      <body name="r_knee"><joint name="r_hip_roll" type="hinge" axis="1 0 0"/></body>
      <body name="r_ankle"><joint name="r_hip_pitch" type="hinge" axis="0 1 0"/></body>
      <body name="r_foot"><joint name="r_knee_pitch" type="hinge" axis="0 1 0"/></body>
      <body name="r_toe"><joint name="r_ankle_pitch" type="hinge" axis="0 1 0"/></body>
      
      <!-- Neck & Head (5) -->
      <body name="n_base"><joint name="neck_yaw" type="hinge" axis="0 0 1"/><geom type="sphere" size="0.03"/></body>
      <body name="n_mid"><joint name="neck_pitch" type="hinge" axis="0 1 0"/></body>
      <body name="n_top"><joint name="neck_roll" type="hinge" axis="1 0 0"/></body>
      <body name="h_yaw"><joint name="head_yaw" type="hinge" axis="0 0 1"/></body>
      <body name="h_pitch"><joint name="head_pitch" type="hinge" axis="0 1 0"/></body>
    </body>
  </worldbody>

  <actuator>
    <!-- We attach 15 virtual motors to the 15 joints -->
    <!-- Left Leg -->
    <motor joint="l_hip_yaw" name="motor_l0"/>
    <motor joint="l_hip_roll" name="motor_l1"/>
    <motor joint="l_hip_pitch" name="motor_l2"/>
    <motor joint="l_knee_pitch" name="motor_l3"/>
    <motor joint="l_ankle_pitch" name="motor_l4"/>
    <!-- Right Leg -->
    <motor joint="r_hip_yaw" name="motor_r5"/>
    <motor joint="r_hip_roll" name="motor_r6"/>
    <motor joint="r_hip_pitch" name="motor_r7"/>
    <motor joint="r_knee_pitch" name="motor_r8"/>
    <motor joint="r_ankle_pitch" name="motor_r9"/>
    <!-- Neck/Head -->
    <motor joint="neck_yaw" name="motor_n10"/>
    <motor joint="neck_pitch" name="motor_n11"/>
    <motor joint="neck_roll" name="motor_n12"/>
    <motor joint="head_yaw" name="motor_n13"/>
    <motor joint="head_pitch" name="motor_n14"/>
  </actuator>
</mujoco>
"""

# Save it to disk just like a real downloaded Hugging Face model
with open("microduck.xml", "w") as f:
    f.write(microduck_mjcf)
    
print("📝 Wrote 'microduck.xml' blueprint to your project folder.")

📝 Wrote 'microduck.xml' blueprint to your project folder.


### Inspecting the Hardware Limits

Now that we have the `microduck.xml` file, we can load it into the `MjModel` (The Blueprint). 

As an AI architect, your job is to ensure the software (the ONNX brain) matches the hardware (the MJCF blueprint). Let's write a quick script to interrogate the MuJoCo model and prove that it has exactly 15 actuators, and that the engine respects our `-1.0` to `1.0` safety limits.

In [ ]:
# 1. Load the blueprint from the file we just created
duck_model = mujoco.MjModel.from_xml_path("microduck.xml")

# 2. Interrogate the physics engine
num_joints = duck_model.njnt
num_actuators = duck_model.nu

print("🦆 MICRODUCK HARDWARE INSPECTION 🦆")
print("-" * 35)
print(f"Total Joints Found:    {num_joints}")
print(f"Total Motors Found:    {num_actuators}")
print("-" * 35)

# 3. Verify the hardware safety clamps
print("Motor Hardware Limits (Control Range):")
for i in range(num_actuators):
    # Retrieve the name and control range of each motor
    motor_name = duck_model.actuator(i).name
    # ctrlrange is an array [min, max]
    min_limit = duck_model.actuator_ctrlrange[i][0]
    max_limit = duck_model.actuator_ctrlrange[i][1]
    
    print(f"  [{i:02d}] {motor_name:<10} : {min_limit:>4.1f} to {max_limit:>4.1f}")

---

# Explain URDF/MJCF and basic of MuJoKo

When scaling a robotics product, standardizing how you define hardware in software is critical. Before a neural network can control a robot, the physics engine needs an exact mathematical blueprint of the hardware—its mass, joint limits, motor torque, and collision boundaries.

This is handled by XML-based modeling languages. The two you must know are **URDF** and **MJCF**.

## 1. The Blueprint Languages: URDF vs. MJCF

### URDF (Unified Robot Description Format)

URDF is the undisputed industry standard created for ROS (Robot Operating System). It defines a robot as a tree of rigid bodies.

* **Links (`<link>`):** The physical parts (e.g., a femur, a tibia). Links contain the visual mesh (what it looks like) and the inertial properties (mass and center of gravity).
* **Joints (`<joint>`):** The hinges that connect two links. A joint defines the axis of rotation and the physical limits (e.g., "this knee can only bend 90 degrees").

URDF is flat. You declare a link, declare another link, and then declare a joint to connect them.

### MJCF (MuJoCo Modeling XML Format)

MJCF is a proprietary XML format built specifically for the **MuJoCo** physics engine. While URDF is great for moving data around in ROS, it lacks the depth required for advanced reinforcement learning (like defining tendons, soft contacts, or specific motor actuators).

* **Nested Structure:** Instead of flat declarations, MJCF nests objects. A `<body name="leg">` physically contains its `<joint>` and `<geom>` (geometry).
* **Simulation-First:** MJCF includes an `<actuator>` section to define the exact electrical limits of the motors, which is essential for training hardware-safe AI.

| Feature | URDF | MJCF |
| --- | --- | --- |
| **Primary Use** | ROS integration, hardware agnostic sharing | DeepMind/MuJoCo reinforcement learning |
| **Structure** | Flat list of links connected by joints | Hierarchical tree of nested bodies |
| **Actuators/Motors** | Very limited | Native support (motors, cylinders, muscles) |
| **Flexibility** | Rigid bodies only | Supports tendons, equality constraints, and soft materials |


## 2. Visualizing the Robot Tree

Whether you use URDF or MJCF, the robot is fundamentally a kinematic tree branching out from a root node (usually the pelvis or base).

> ![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQkhDuU6op1lcLfQ8iZKpdgCQwAAX0RM3HN7AyyLvNDEh0oCYeLZQ5HWps&s=10)   
A Standard kinematic robot tree. Source: Articulated Robotics  

> **Key insight:** The tree flows in one direction. If the "Base" moves, the math automatically calculates the cascading positional updates for every child node down to the "Camera."


## 3. MuJoCo Physics Simulator Basics

**MuJoCo** (Multi-Joint dynamics with Contact) is the physics engine that reads these blueprints. Originally developed by Emo Todorov and acquired by DeepMind, it is optimized for the continuous control problems found in robotics and biomechanics.

Here are the core concepts the engine calculates every few milliseconds:

* **Forward Dynamics:** The engine asks, *"Given the current state of the robot and the motor forces just applied, what is the exact position and velocity of every joint 2 milliseconds from now?"* This is the core loop used in Reinforcement Learning.
* **Inverse Dynamics:** The engine asks, *"To make the duck's foot reach this exact XYZ coordinate, how much torque must be applied to the hip, knee, and ankle?"*
* **Degrees of Freedom (DOF):** The number of independent ways the robot can move. A free-floating body in space has 6 DOF (X, Y, Z, Roll, Pitch, Yaw). A simple hinge joint adds 1 DOF. The 15-DOF Microduck has 15 independent motor-controlled hinges.
* **Soft Contacts:** Traditional game engines (like Unity or Unreal) use "hard" contacts—if two objects touch, they violently bounce off each other. MuJoCo uses mathematically "soft" contacts, allowing objects to deform slightly and push against each other realistically. This prevents the simulation from exploding when a heavy robot steps on a solid floor.

![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT_88FB0B6ZX5Q4A0QSQRdQnwYYoJtXjDZnmRsswJyaHMZggCdX-ALnsyY&s=10)   
MuJoCo's diagnostic interface. Source: Medium

# Converting a URDF to an MJCF  

Converting a URDF to an MJCF is the "Google Translate" of robotics. Hardware manufacturers almost always provide URDF files because they are universally accepted, but URDFs lack the physics depth (like motors, friction, and soft contacts) required for advanced AI training.

Fortunately, DeepMind built a native translator directly into MuJoCo.

Here is how you perform the conversion in your `uv` environment, along with the most critical "gotcha" in the process.

## The Two-Step Conversion Strategy

There are two ways to do this. We will use the **Wrapper Method** because it is the industry standard for Physical AI.

Instead of permanently altering the manufacturer's original URDF, we create an MJCF "wrapper" that dynamically imports the URDF and injects our custom physics rules and motors on top of it.

### Step 1: Create the Wrapper MJCF

Imagine the manufacturer gave you a file named `microduck_vendor.urdf` that only contains the bones and weight of the robot.

Create a new file named `microduck_wrapper.xml` and use the `<include>` tag. This tells MuJoCo to load the URDF, apply a compiler to optimize the meshes, and then attach the motors that the URDF format fundamentally lacks.

```xml
<mujoco model="microduck_optimized">
  <!-- 1. The Compiler: Optimizes the imported URDF for MuJoCo -->
  <compiler angle="radian" meshdir="meshes/" autolimits="true"/>
  
  <!-- 2. The Import: Bring in the manufacturer's raw URDF -->
  <include file="microduck_vendor.urdf"/>
  
  <!-- 3. The Injection: Add MuJoCo-specific physics and hardware safety -->
  <default>
    <motor ctrlrange="-1.0 1.0" ctrllimited="true"/> 
  </default>
  
  <!-- 4. The Muscles: URDF doesn't have motors, so we add them here -->
  <actuator>
    <motor joint="l_hip_yaw" name="motor_l0"/>
    <motor joint="r_hip_yaw" name="motor_r0"/>
    <!-- ... rest of your 15 motors ... -->
  </actuator>
</mujoco>

```

### Step 2: The Python Compiler Script

Now, we write a quick Python script to process this wrapper. When MuJoCo loads a wrapper, it automatically merges the URDF and the MJCF into a single, highly optimized mathematical graph in its memory.

We can then command MuJoCo to dump that merged memory out into a single, clean, finalized MJCF file.

Create a file named `compile_urdf.py`:

```python
import mujoco

def main():
    print("🔄 Loading the URDF through the MJCF Wrapper...")
    
    # 1. Load the wrapper (which automatically pulls in the URDF)
    # Note: This assumes you have a mock 'microduck_vendor.urdf' in the same folder
    try:
        model = mujoco.MjModel.from_xml_path("microduck_wrapper.xml")
    except Exception as e:
        print(f"[-] Error loading model: {e}")
        print("[!] Ensure you have the vendor URDF and mesh files in the correct directory.")
        return

    print("✅ Successfully compiled the URDF into MuJoCo's memory.")
    
    # 2. Dump the highly optimized, merged model into a final MJCF file
    output_filename = "microduck_final_compiled.xml"
    mujoco.mj_saveLastXML(output_filename, model)
    
    print(f"💾 Saved the fully optimized MJCF as: {output_filename}")
    
    # 3. Quick sanity check
    print("-" * 40)
    print("🔍 Final Compiled Hardware Check:")
    print(f"   Joints imported from URDF: {model.njnt}")
    print(f"   Motors injected from MJCF: {model.nu}")
    print("-" * 40)

if __name__ == "__main__":
    main()

```

## The "Gotcha": Autolimits and Inertia

When you run this conversion on a real manufacturer's URDF, you will often see warnings in your terminal about "inertia" or "joint limits."

Because URDF was originally built for moving data rather than simulating physics, manufacturers sometimes leave the physical mass or friction values blank, or set them to `0.0`. If a robot's leg weighs `0.0` kg, the physics engine will divide by zero and the simulation will explode (the duck will vanish into the sky at infinite speed).

MuJoCo's `<compiler>` tag acts as an automated mechanic. By adding `autolimits="true"`, MuJoCo will automatically scan the imported URDF for missing physics data, infer the correct values based on the volume of the 3D meshes, and quietly fix the math before saving the final MJCF.

## How do I convert an existing URDF file provided by a hardware manufacturer into an MJCF file optimized for MuJoCo simulation?  

It looks like we might have double-tapped that question! Since we just walked through the **Wrapper Method** for converting URDFs into MJCFs, let's take that newly converted blueprint and give it the one thing most manufacturer URDFs completely lack: **Vision**.

Once your physical robot is converted into a MuJoCo-optimized MJCF, adding a virtual camera allows your neural network to use computer vision (like navigating a room or finding a charging station) alongside its physical balance sensors.

Here is how you inject a virtual RGB camera into your `microduck_wrapper.xml`.

## 1. Injecting the Camera Tag

In MuJoCo, cameras are attached directly to specific physical bodies in the XML tree. If we want the camera to move realistically as the duck walks, we attach it to the `h_pitch` (head pitch) body we defined earlier.

Update your MJCF wrapper to include the `<camera>` tag:

```xml
<mujoco model="microduck_vision">
  <compiler angle="radian" autolimits="true"/>
  <include file="microduck_vendor.urdf"/>
  
  <worldbody>
    <!-- Assuming the included URDF has a body named 'head' -->
    <body name="head">
      <!-- We attach a 60-degree Field of View (FOV) camera to the head -->
      <camera name="eye_cam" pos="0.05 0 0" zaxis="1 0 0" fovy="60"/>
    </body>
  </worldbody>

  <!-- ... (Motors and actuators remain the same) ... -->
</mujoco>

```

* **`pos="0.05 0 0"`**: Pushes the camera 5cm forward so it doesn't accidentally render the inside of the duck's own head.
* **`zaxis="1 0 0"`**: Points the camera lens forward along the X-axis.

## 2. Rendering the Pixels in Python

Once the camera is in the XML, your Python control loop can render the pixels. Note that rendering graphics takes much more GPU power than calculating pure physics, so we typically render cameras at a lower frequency (e.g., 10Hz) than the balance loop (50Hz).

Here is how you extract the pixel array from the MuJoCo engine:

```python
import mujoco
import numpy as np

# 1. Load the model and create a rendering context
model = mujoco.MjModel.from_xml_path("microduck_vision.xml")
data = mujoco.MjData(model)

# Instantiate the MuJoCo OpenGL renderer
# We set a low resolution (e.g., 224x224) standard for AI vision models
renderer = mujoco.Renderer(model, height=224, width=224)

# 2. Step the physics engine once
mujoco.mj_step(model, data)

# 3. Render the camera frame
renderer.update_scene(data, camera="eye_cam")
pixels = renderer.render()

# 'pixels' is now a standard NumPy array of shape (224, 224, 3) containing RGB values.
print(f"📷 Captured frame shape: {pixels.shape}")
print(f"🎨 Data type: {pixels.dtype} (Ready for PyTorch/TorchVision!)")

```

Because the `pixels` variable is a standard NumPy array, you can pass it directly into an image-processing neural network (like a ResNet or a Vision Transformer) to let your Microduck "see" obstacles before it physically trips over them.

## Want to know how to fuse vision with the balance loop?

How do I combine the 10Hz camera pixel data with the 50Hz physical IMU data in my neural network architecture?


Copy and paste this text into a **Markdown cell** in your Jupyter Notebook, then hit `Shift + Enter`.

```markdown
# Module 6: The Spinal Cord vs. The Visual Cortex (Sensor Fusion)

If you trip on a rock while walking, you do not wait for your eyes to focus, identify the rock, and tell your brain what it is. If you waited for your eyes, you would hit the ground. Instead, your inner ear (IMU) and leg muscles detect the tilt instantly, and your spinal cord fires a reflex to catch your balance. Your eyes figure out what happened *after* you are already safe.

In Physical AI, we must replicate this biological asynchronous design. 

*   **The Spinal Cord (IMU & Encoders):** Runs strictly at 50Hz (20ms). It must never wait for anything.
*   **The Visual Cortex (Camera):** Runs at 10Hz (100ms). It is heavy and slow.

### The Architecture: Shared Memory Buffer
We run the vision system in a background thread. When it finishes processing an image, it compresses the picture into a tiny list of numbers (a "latent vector" or "visual feature") and saves it to a shared memory box. The 50Hz balance loop constantly grabs the *most recent* IMU data, peeks at the *most recently saved* visual feature, combines them, and fires the motors.

```

Next, add a **Code cell** below it to simulate this asynchronous threading in Python. This is the exact pattern used in production robotics to prevent high-latency sensors from crashing the low-latency balance loop.

```python
import time
import threading
import numpy as np

print("🧠 Booting Dual-Loop Architecture...")

# 1. The Shared Memory Box (Thread-safe)
# We store the compressed visual representation here (e.g., a 32-element array).
class SharedMemory:
    def __init__(self):
        self.lock = threading.Lock()
        # Initialize with zeros until the camera processes its first frame
        self.latest_vision_features = np.zeros(32, dtype=np.float32)

    def update_vision(self, new_features):
        with self.lock:
            self.latest_vision_features = new_features

    def get_vision(self):
        with self.lock:
            return self.latest_vision_features.copy()

shared_memory = SharedMemory()

# 2. The Visual Cortex (Background Thread - 10Hz)
def camera_loop():
    print("  [Camera] Vision thread started (10Hz).")
    for _ in range(5):
        time.sleep(0.1)  # Simulate 100ms camera capture and CNN inference latency
        
        # Simulate compressing a 224x224 RGB image into 32 useful features
        new_features = np.random.uniform(-1.0, 1.0, 32).astype(np.float32)
        shared_memory.update_vision(new_features)
        print(f"  [Camera] 📷 New visual features encoded and saved to shared memory.")

vision_thread = threading.Thread(target=camera_loop)
vision_thread.start()

# 3. The Spinal Cord (Main Loop - 50Hz)
print("  [Spinal Cord] Balance loop started (50Hz).")
for tick in range(1, 26):
    start_time = time.time()
    
    # 3a. Read fast 50Hz physical sensors (IMU, joints)
    imu_data = np.random.uniform(-0.1, 0.1, 15).astype(np.float32)
    
    # 3b. Read the LAST KNOWN vision features (Instantaneous, no waiting!)
    vision_data = shared_memory.get_vision()
    
    # 3c. FUSION: Concatenate the two streams into one array for the Actor network
    fused_observations = np.concatenate([imu_data, vision_data])
    
    # In reality, this fused array goes into your ONNX session here
    
    if tick % 5 == 0:
        print(f"    [Spinal Cord] Tick {tick:02d} | Fused input size: {len(fused_observations)} floats | Kept balance!")
        
    # Maintain strict 50Hz timing
    elapsed = time.time() - start_time
    time.sleep(max(0.0, 0.02 - elapsed))

vision_thread.join()
print("🏁 Dual-Loop Simulation Complete.")

```

Add one more **Markdown cell** to explain how the PyTorch neural network is designed to receive this "fused" data array:

```markdown
### The Sensor Fusion PyTorch Architecture

When we train this in Stable Baselines3, the neural network inside the Actor looks like a "Y" shape. 

1.  **Vision Encoder (CNN/ResNet):** Takes the `(224, 224, 3)` image and compresses it down to `32` numbers. This is computationally expensive, which is why your RTX 5060 handles it on the background thread.
2.  **Proprioception Encoder (MLP):** Takes the `15` IMU/Joint numbers and processes them.
3.  **The Fusion Layer:** Joins the `32` vision numbers and `15` physical numbers together, passing the combined `47` numbers to the final motor-control layer.

```

Finally, add this **Code cell** to show the PyTorch representation of that "Y" shaped brain:

```python
import torch
import torch.nn as nn

class SensorFusionActor(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 1. The Vision Pathway (Compresses image to 32 features)
        # (Mocked as a simple linear layer here for readability, usually a CNN)
        self.vision_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(224 * 224 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )
        
        # 2. The Fusion & Motor Pathway 
        # Takes the 32 vision features + 15 IMU sensors = 47 total inputs
        self.action_network = nn.Sequential(
            nn.Linear(32 + 15, 64),
            nn.ReLU(),
            nn.Linear(64, 15),
            nn.Tanh() # Hardware safety clamp (-1.0 to 1.0)
        )

    def forward(self, image, imu_sensors):
        # The background thread does this:
        visual_features = self.vision_encoder(image)
        
        # The 50Hz balance loop does this:
        # Fuse them side-by-side
        fused_state = torch.cat([visual_features, imu_sensors], dim=1)
        
        # Output final motor commands
        motor_commands = self.action_network(fused_state)
        return motor_commands

print("🧠 PyTorch Sensor Fusion architecture defined.")

```

## Want to put everything together?

A complete, final `main.py` script that combines the  
> dual-loop sensor fusion,   
> the ONNX inference, and   
> the headless MuJoCo physics engine 

>> into one production-ready file?

Overwrite your `main.py` with this final architecture:

import time
import threading
import numpy as np
import mujoco
import onnxruntime as ort
import os

# ---------------------------------------------------------
# 1. Thread-Safe Memory for Asynchronous Sensor Fusion
# ---------------------------------------------------------
class SharedMemory:
    def __init__(self):
        self.lock = threading.Lock()
        # Initializing with 32 zeros (mocking a compressed visual feature vector)
        self.latest_vision = np.zeros(32, dtype=np.float32)
        self.running = True

    def update_vision(self, new_features):
        with self.lock:
            self.latest_vision = new_features

    def get_vision(self):
        with self.lock:
            return self.latest_vision.copy()

    def stop(self):
        with self.lock:
            self.running = False

    def is_running(self):
        with self.lock:
            return self.running

# ---------------------------------------------------------
# 2. The Visual Cortex (10Hz Background Thread)
# ---------------------------------------------------------
def vision_loop(shared_mem):
    print("  [Camera] Vision thread initialized (10Hz).")
    
    # Simulating 10Hz (100ms) cadence for capturing and processing RGB frames
    target_hz = 10
    tick_time = 1.0 / target_hz
    
    while shared_mem.is_running():
        start_time = time.time()
        
        # In a real robot, you capture a frame here and run a PyTorch/ONNX Vision model.
        # We simulate the resulting 32-element feature vector:
        new_features = np.random.uniform(-1.0, 1.0, 32).astype(np.float32)
        shared_mem.update_vision(new_features)
        
        elapsed = time.time() - start_time
        time.sleep(max(0.0, tick_time - elapsed))

# ---------------------------------------------------------
# 3. The Spinal Cord (50Hz Main Physics & Control Loop)
# ---------------------------------------------------------
def main():
    print("🚀 Booting Microduck Production Edge Controller...")
    
    # Safely load the MuJoCo blueprint (Fallback to mock if missing)
    xml_path = "microduck.xml"
    if os.path.exists(xml_path):
        model = mujoco.MjModel.from_xml_path(xml_path)
        print(f"🌍 MuJoCo loaded '{xml_path}' ({model.nu} actuators found).")
    else:
        print(f"⚠️ '{xml_path}' not found. Generating headless mock universe.")
        mock_xml = """<mujoco><worldbody><geom type="plane" size="1 1 0.1"/><body pos="0 0 1"><joint type="free"/><geom type="capsule" size="0.1 0.2"/></body></worldbody><actuator><motor joint="0" name="mock"/></actuator></mujoco>"""
        model = mujoco.MjModel.from_xml_string(mock_xml)
        
    data = mujoco.MjData(model)

    # Safely load the ONNX Reflex Policy (Fallback to mock inference if missing)
    onnx_path = "microduck_walking_policy.onnx"
    use_real_onnx = os.path.exists(onnx_path)
    if use_real_onnx:
        session = ort.InferenceSession(onnx_path)
        print(f"🧠 ONNX Policy '{onnx_path}' loaded into memory.")
    else:
        print(f"⚠️ '{onnx_path}' not found. Simulating inference outputs.")

    # Initialize shared memory and start the vision thread
    shared_mem = SharedMemory()
    v_thread = threading.Thread(target=vision_loop, args=(shared_mem,))
    v_thread.start()

    print("⚡ Starting 50Hz Physics & Control Loop...")
    print("-" * 55)

    target_hz = 50
    tick_time = 1.0 / target_hz
    
    try:
        # Run the loop for 100 ticks (2 seconds of physical time)
        for tick in range(1, 101):
            start_time = time.time()
            
            # [SENSE] Read the 15 proprioceptive sensors directly from MuJoCo
            # qpos = joint positions, qvel = joint velocities. We mock 15 floats here:
            imu_data = np.random.uniform(-0.1, 0.1, 15).astype(np.float32)
            
            # Read the latest vision features instantly from the background thread
            vision_data = shared_mem.get_vision()
            
            # Fuse the data (32 vision + 15 IMU = 47 inputs for the neural network)
            fused_observation = np.concatenate([vision_data, imu_data]).reshape(1, -1)
            
            # [THINK] Execute the low-latency reflexes
            if use_real_onnx:
                # Actual execution on your local hardware
                actions = session.run(None, {"observations": fused_observation})[0][0]
            else:
                # Mock output strictly clamped between -1.0 and 1.0
                actions = np.random.uniform(-1.0, 1.0, model.nu).astype(np.float32)
            
            # [ACT] Apply the motor commands to MuJoCo and step physics forward
            data.ctrl[:model.nu] = actions[:model.nu]
            mujoco.mj_step(model, data)
            
            # Log telemetry every 10 ticks
            if tick % 10 == 0:
                altitude = data.qpos[2] if len(data.qpos) > 2 else 0.0
                print(f"Tick {tick:03d} | Z-Altitude: {altitude:>5.2f}m | Motors Fired: {model.nu}")
            
            # Enforce 50Hz execution timing
            elapsed = time.time() - start_time
            time.sleep(max(0.0, tick_time - elapsed))
            
    except KeyboardInterrupt:
        print("\n🛑 Manual override triggered. Shutting down...")
    finally:
        # Clean up the background thread
        shared_mem.stop()
        v_thread.join()
        print("-" * 55)
        print("✅ Graceful shutdown complete. Hardware safe.")

if __name__ == "__main__":
    main()

## integrate the Antigravity IDE?  

Guide to installing and setting up the Antigravity IDE on Windows and connecting it to WSL2 environment to manage this project.


As an AI startup founder working on complex DevSecOps and Physical AI pipelines, standard text editors eventually become bottlenecks.

The **Google Antigravity IDE** is built specifically for this transition. Unlike traditional editors that just highlight syntax, Antigravity embeds an autonomous AI agent directly into your workspace. It can read your Python scripts, see your terminal output, write boilerplate, and execute `uv` commands asynchronously while you focus on the core architecture.

Because you are running Windows 11 with an Ubuntu WSL2 subsystem, the setup is straightforward but requires a specific connection method to ensure the agent has full access to your Linux environment.

Here is the step-by-step guide to installing Antigravity and wiring it into your WSL2 workspace.

## 1. Install the Antigravity CLI and App on Windows

You need to install the Windows host application first. Do this from your standard **Windows PowerShell** (not your Ubuntu terminal).

```powershell
# 1. Open Windows PowerShell as Administrator
# 2. Install the Antigravity App via Winget
winget install Google.Antigravity

```

This installs the Antigravity GUI on your Windows desktop.

## 2. Install the WSL2 Connector

If you just open Antigravity on Windows, it will try to manage your Windows filesystem. We need it to connect directly into your `~/agy_projects/` directory inside Ubuntu.

Open your **Ubuntu WSL2 Terminal** and install the Antigravity CLI specifically for Linux:

```bash
# 1. Download and install the Antigravity CLI for Linux
curl -sL https://antigravity.google.com/install.sh | bash

# 2. Verify the installation
agy --version

```

## 3. Launch Antigravity from WSL2

Just like launching VS Code from WSL (`code .`), you will launch Antigravity directly from your Linux project folder. This tells the Windows app to connect via the WSL bridge.

In your **Ubuntu WSL2 Terminal**, navigate to your Microduck project and launch the IDE:

```bash
cd ~/agy_projects/physical_ai/microduck_sim/
agy .

```

The Antigravity GUI will open on your Windows desktop, but you will notice a green "WSL: Ubuntu" badge in the bottom corner. The IDE, and its embedded agent, are now securely sandboxed inside your Linux subsystem.

---

## 4. Configuring the Agent Workspace

Once Antigravity opens, you need to introduce the agent to your specific `uv` environment so it doesn't accidentally try to use standard `pip` or global Python paths.

1. **Open the Agent Sidebar:** Press `Ctrl + J` to open the Antigravity Agent chat panel.
2. **Set the Context:** Type the following command to the agent to permanently configure its behavior for this project:

> *"This project uses `uv` for environment management. Never use pip. The Python executable is located at `.venv/bin/python`. My environment variables are located in the parent directory at `../.env`."*

## 5. Test the Agent

Let's test the agent's ability to read your filesystem and execute terminal commands. Ask the agent in the sidebar:

> *"Please run the `main.py` script and tell me if the 50Hz control loop executes successfully."*

The agent will autonomously open an integrated terminal, activate your `uv` environment, run `uv run main.py`, read the telemetry outputs, and report back to you in the chat.

You now have a fully integrated, AI-assisted development environment perfectly suited for managing complex robotics pipelines.